# Assemble Data

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.ndimage import uniform_filter1d  # simple smoothing
import re

In [2]:
pd.set_option('display.max_columns', None)  # show all columns
# pd.set_option('display.max_rows', None)  # show all columns
# import os
# os.chdir('..')

### Start with plot features

In [3]:
# get elevation features
df = pd.read_pickle('../data/plot_elev_features.pkl')

In [4]:
df

,plot_id,slope_mean,slope_min,slope_max,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,aspect_mean,aspect_min,aspect_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,geometry,total_relief,area_m2,area_ha
0,0,5.020058,3.925700,6.804783,0.000489,-0.009361,0.013225,0.000620,-0.010633,0.013191,0.000489,-0.009361,0.013225,175.379315,155.373535,190.393188,207.326294,208.851974,208.022307,22.362457,23.888138,23.058465,"POLYGON ((-119.78208 45.87245, -119.78215 45.8...",1.525681,451.141838,0.045114
1,1,5.289605,3.356515,8.816703,-0.000258,-0.014769,0.013737,-0.001123,-0.016449,0.012478,-0.000258,-0.014769,0.013737,169.123825,144.559204,188.072983,206.874710,208.469513,207.776839,21.910873,23.505676,22.812984,"POLYGON ((-119.78179 45.87245, -119.78167 45.8...",1.594803,381.909845,0.038191
2,2,7.150476,4.206255,11.814886,0.000065,-0.030229,0.034444,-0.000754,-0.043223,0.027175,0.000065,-0.030229,0.034444,152.046614,135.318848,175.093628,205.507690,207.790955,206.759655,20.543854,22.827118,21.795817,"POLYGON ((-119.78139 45.87246, -119.78129 45.8...",2.283264,304.757142,0.030476
3,3,9.178217,5.255136,12.657069,0.000241,-0.022650,0.018867,0.000610,-0.020701,0.012653,0.000241,-0.022650,0.018867,175.202423,159.699402,189.218704,210.499161,214.308624,212.434447,25.535324,29.344788,27.470613,"POLYGON ((-119.78299 45.87287, -119.78300 45.8...",3.809464,416.261511,0.041626
4,4,6.641992,4.568419,10.110537,0.000753,-0.013419,0.014943,0.000611,-0.011132,0.012551,0.000753,-0.013419,0.014943,173.372281,156.050049,189.273071,209.319901,213.504730,211.261849,24.356064,28.540894,26.298020,"POLYGON ((-119.78263 45.87269, -119.78269 45.8...",4.184830,934.859055,0.093486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3593,3593,6.524146,4.537724,8.501439,-0.001274,-0.024254,0.019990,-0.001476,-0.019637,0.020838,-0.001274,-0.024254,0.019990,163.878749,140.306885,186.459793,228.098480,232.209412,230.327266,43.134644,47.245575,45.363431,"POLYGON ((-119.77610 45.87519, -119.77626 45.8...",4.110931,544.855651,0.054486
3594,3594,3.070807,1.769737,4.388859,0.000577,-0.013051,0.017839,0.000672,-0.015131,0.010445,0.000577,-0.013051,0.017839,176.192068,142.778198,228.122147,234.638351,235.519440,235.030851,49.674515,50.555603,50.067012,"POLYGON ((-119.77718 45.87568, -119.77718 45.8...",0.881088,319.126138,0.031913
3595,3595,3.290681,1.204210,6.403341,-0.001655,-0.029511,0.026497,-0.002227,-0.026850,0.022847,-0.001655,-0.029511,0.026497,210.650471,141.063812,265.084839,235.022568,236.736877,235.759766,50.058731,51.773041,50.795942,"POLYGON ((-119.77685 45.87573, -119.77680 45.8...",1.714310,591.653344,0.059165
3596,3596,6.540738,1.204210,9.352532,-0.000439,-0.029511,0.027310,-0.001295,-0.025244,0.019796,-0.000439,-0.029511,0.027310,140.301358,110.309143,226.955688,232.924850,236.760712,235.060714,47.961014,51.796875,50.096894,"POLYGON ((-119.77652 45.87579, -119.77642 45.8...",3.835861,847.012539,0.084701


### Compute vectors for aspect and slope, and some interaction terms

In [5]:
# Aspect: convert to radians and compute sin/cos
df['aspect_min_cos'] = np.cos(np.radians(df['aspect_min']))
df['aspect_min_sin'] = np.sin(np.radians(df['aspect_min']))

df['aspect_max_cos'] = np.cos(np.radians(df['aspect_max']))
df['aspect_max_sin'] = np.sin(np.radians(df['aspect_max']))

df['aspect_mean_cos'] = np.cos(np.radians(df['aspect_mean']))
df['aspect_mean_sin'] = np.sin(np.radians(df['aspect_mean']))

# Drop raw aspect values
df = df.drop(columns=['aspect_min', 'aspect_max', 'aspect_mean'])

df['slope_rad'] = np.radians(df['slope_mean'])
df['slope_grad'] = np.tan(df['slope_rad'])


df['slope_x'] = df['slope_grad'] * df['aspect_mean_cos']
df['slope_y'] = df['slope_grad'] * df['aspect_mean_sin']

df = df.drop(columns = ['slope_mean','slope_min','slope_max'])

In [6]:
df

,plot_id,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,geometry,total_relief,area_m2,area_ha,aspect_min_cos,aspect_min_sin,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_rad,slope_grad,slope_x,slope_y
0,0,0.000489,-0.009361,0.013225,0.000620,-0.010633,0.013191,0.000489,-0.009361,0.013225,207.326294,208.851974,208.022307,22.362457,23.888138,23.058465,"POLYGON ((-119.78208 45.87245, -119.78215 45.8...",1.525681,451.141838,0.045114,-0.909044,0.416701,-0.983593,-0.180402,-0.996750,0.080559,0.087617,0.087841,-0.087556,0.007076
1,1,-0.000258,-0.014769,0.013737,-0.001123,-0.016449,0.012478,-0.000258,-0.014769,0.013737,206.874710,208.469513,207.776839,21.910873,23.505676,22.812984,"POLYGON ((-119.78179 45.87245, -119.78167 45.8...",1.594803,381.909845,0.038191,-0.814715,0.579861,-0.990090,-0.140434,-0.982037,0.188687,0.092321,0.092584,-0.090921,0.017469
2,2,0.000065,-0.030229,0.034444,-0.000754,-0.043223,0.027175,0.000065,-0.030229,0.034444,205.507690,207.790955,206.759655,20.543854,22.827118,21.795817,"POLYGON ((-119.78139 45.87246, -119.78129 45.8...",2.283264,304.757142,0.030476,-0.711031,0.703161,-0.996336,0.085528,-0.883329,0.468753,0.124799,0.125451,-0.110815,0.058806
3,3,0.000241,-0.022650,0.018867,0.000610,-0.020701,0.012653,0.000241,-0.022650,0.018867,210.499161,214.308624,212.434447,25.535324,29.344788,27.470613,"POLYGON ((-119.78299 45.87287, -119.78300 45.8...",3.809464,416.261511,0.041626,-0.937885,0.346945,-0.987084,-0.160203,-0.996496,0.083636,0.160190,0.161575,-0.161008,0.013513
4,4,0.000753,-0.013419,0.014943,0.000611,-0.011132,0.012551,0.000753,-0.013419,0.014943,209.319901,213.504730,211.261849,24.356064,28.540894,26.298020,"POLYGON ((-119.78263 45.87269, -119.78269 45.8...",4.184830,934.859055,0.093486,-0.913900,0.405938,-0.986932,-0.161140,-0.993317,0.115418,0.115925,0.116447,-0.115669,0.013440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3593,3593,-0.001274,-0.024254,0.019990,-0.001476,-0.019637,0.020838,-0.001274,-0.024254,0.019990,228.098480,232.209412,230.327266,43.134644,47.245575,45.363431,"POLYGON ((-119.77610 45.87519, -119.77626 45.8...",4.110931,544.855651,0.054486,-0.769476,0.638675,-0.993651,-0.112506,-0.960676,0.277671,0.113868,0.114363,-0.109865,0.031755
3594,3594,0.000577,-0.013051,0.017839,0.000672,-0.015131,0.010445,0.000577,-0.013051,0.017839,234.638351,235.519440,235.030851,49.674515,50.555603,50.067012,"POLYGON ((-119.77718 45.87568, -119.77718 45.8...",0.881088,319.126138,0.031913,-0.796300,0.604902,-0.667545,-0.744570,-0.997792,0.066412,0.053596,0.053647,-0.053529,0.003563
3595,3595,-0.001655,-0.029511,0.026497,-0.002227,-0.026850,0.022847,-0.001655,-0.029511,0.026497,235.022568,236.736877,235.759766,50.058731,51.773041,50.795942,"POLYGON ((-119.77685 45.87573, -119.77680 45.8...",1.714310,591.653344,0.059165,-0.777846,0.628454,-0.085681,-0.996323,-0.860293,-0.509799,0.057433,0.057496,-0.049464,-0.029312
3596,3596,-0.000439,-0.029511,0.027310,-0.001295,-0.025244,0.019796,-0.000439,-0.029511,0.027310,232.924850,236.760712,235.060714,47.961014,51.796875,50.096894,"POLYGON ((-119.77652 45.87579, -119.77642 45.8...",3.835861,847.012539,0.084701,-0.347085,0.937834,-0.682564,-0.730826,-0.769415,0.638750,0.114157,0.114656,-0.088218,0.073236


## Now add NDVI for each plot to features

### open up the filtered and smoothed ndvi df

In [7]:
veg_agg = pd.read_pickle('../data/ndvi/plots/final_df.pkl')
keep_cols = ['plot_id', 'year']

In [8]:
veg_agg = veg_agg.dropna(axis = 1)
veg_agg

,plot_id,year,ndvi_smooth_mean_13,ndvi_smooth_mean_14,ndvi_smooth_mean_15,ndvi_smooth_mean_16,ndvi_smooth_mean_17,ndvi_smooth_mean_18,ndvi_smooth_mean_19,ndvi_smooth_mean_20,ndvi_smooth_mean_21,ndvi_smooth_mean_22,ndvi_smooth_mean_23,ndvi_smooth_mean_24,ndvi_smooth_mean_25,ndvi_smooth_mean_26,ndvi_smooth_mean_27,ndvi_smooth_mean_28,ndvi_smooth_mean_29,ndvi_smooth_mean_30,ndvi_smooth_mean_31,ndvi_smooth_mean_32,ndvi_smooth_mean_33,ndvi_smooth_mean_34,ndvi_smooth_mean_35,ndvi_smooth_mean_36,ndvi_smooth_mean_37,ndvi_smooth_mean_38,ndvi_smooth_mean_39,ndvi_smooth_mean_40,ndvi_smooth_mean_41,ndvi_smooth_mean_42,ndvi_smooth_mean_43,ndvi_smooth_mean_44,ndvi_smooth_mean_45,ndvi_smooth_slope_1,ndvi_smooth_slope_2,ndvi_smooth_slope_3,ndvi_smooth_slope_4,ndvi_smooth_slope_5,ndvi_smooth_slope_6,ndvi_smooth_slope_7,ndvi_smooth_slope_8,ndvi_smooth_slope_9,ndvi_smooth_slope_10,ndvi_smooth_slope_11,ndvi_smooth_slope_12,ndvi_smooth_slope_13,ndvi_smooth_slope_14,ndvi_smooth_slope_15,ndvi_smooth_slope_16,ndvi_smooth_slope_17,ndvi_smooth_slope_18,ndvi_smooth_slope_19,ndvi_smooth_slope_20,ndvi_smooth_slope_21,ndvi_smooth_slope_22,ndvi_smooth_slope_23,ndvi_smooth_slope_24,ndvi_smooth_slope_25,ndvi_smooth_slope_26,ndvi_smooth_slope_27,ndvi_smooth_slope_28,ndvi_smooth_slope_29,ndvi_smooth_slope_30,ndvi_smooth_slope_31,ndvi_smooth_slope_32,ndvi_smooth_slope_33,ndvi_smooth_slope_34,ndvi_smooth_slope_35,ndvi_smooth_slope_36,ndvi_smooth_slope_37,ndvi_smooth_slope_38,ndvi_smooth_slope_39,ndvi_smooth_slope_40,ndvi_smooth_slope_41,ndvi_smooth_slope_42,ndvi_smooth_slope_43,ndvi_smooth_slope_44,ndvi_smooth_slope_45,ndvi_smooth_slope_46,ndvi_smooth_slope_47,ndvi_smooth_slope_48,ndvi_smooth_slope_49,ndvi_smooth_slope_50,ndvi_smooth_slope_51,ndvi_smooth_slope_52,ndvi_smooth_std_1,ndvi_smooth_std_2,ndvi_smooth_std_3,ndvi_smooth_std_4,ndvi_smooth_std_5,ndvi_smooth_std_6,ndvi_smooth_std_7,ndvi_smooth_std_8,ndvi_smooth_std_9,ndvi_smooth_std_10,ndvi_smooth_std_11,ndvi_smooth_std_12,ndvi_smooth_std_13,ndvi_smooth_std_14,ndvi_smooth_std_15,ndvi_smooth_std_16,ndvi_smooth_std_17,ndvi_smooth_std_18,ndvi_smooth_std_19,ndvi_smooth_std_20,ndvi_smooth_std_21,ndvi_smooth_std_22,ndvi_smooth_std_23,ndvi_smooth_std_24,ndvi_smooth_std_25,ndvi_smooth_std_26,ndvi_smooth_std_27,ndvi_smooth_std_28,ndvi_smooth_std_29,ndvi_smooth_std_30,ndvi_smooth_std_31,ndvi_smooth_std_32,ndvi_smooth_std_33,ndvi_smooth_std_34,ndvi_smooth_std_35,ndvi_smooth_std_36,ndvi_smooth_std_37,ndvi_smooth_std_38,ndvi_smooth_std_39,ndvi_smooth_std_40,ndvi_smooth_std_41,ndvi_smooth_std_42,ndvi_smooth_std_43,ndvi_smooth_std_44,ndvi_smooth_std_45,ndvi_smooth_std_46,ndvi_smooth_std_47,ndvi_smooth_std_48,ndvi_smooth_std_49,ndvi_smooth_std_50,ndvi_smooth_std_51,ndvi_smooth_std_52,ndvi_cov,ndvi_mean,ndvi_std,ndvi_mean_norm,ndvi_std_norm,ndvi_cov_norm,health
0,0.0,2016,0.417441,0.417741,0.415264,0.412015,0.410159,0.410159,0.408649,0.406006,0.403366,0.400812,0.398296,0.395780,0.393264,0.390748,0.388232,0.384497,0.373238,0.356197,0.348879,0.346145,0.345504,0.355702,0.381068,0.413351,0.438675,0.443236,0.440521,0.432348,0.424176,0.416003,0.407830,0.379300,0.366964,0.0,0.000000,0.000000,0.000000,0.000000,0.000076,0.000076,0.000076,0.000076,0.000076,0.000076,0.000076,0.000076,-0.000078,-0.000464,-0.000464,0.000000,0.000000,-0.000378,-0.000378,-0.000376,-0.000359,-0.000359,-0.000359,-0.000359,-0.000359,-0.000359,-0.001071,-0.001783,-0.001851,-0.000534,-0.000351,0.000406,0.003347,0.003689,0.005842,0.000652,0.000652,-0.001168,-0.001168,-0.001168,-0.001168,-0.001168,-0.008094,0.000948,0.000948,0.000948,0.001134,0.001319,0.001319,-0.005969,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000164,0.000164,0.000164,0.000164,0.000164,0.000164,0.000164,0.000164,0.000313,0.001003,0.001003,0.000000,0.000000,0.000816,0.000816,0.000811,0.000776,0.000776,0.000776,0.000776,0.000776,0.000776,0.002445,0.003851,0.004258,0.001196,0.000759,0.000913,0.007293,0.007969,0.012621,0.001408,0.001408,0.002522,0.002522,0.002522,0.002522,0.002522,0.018546,

In [9]:
df = df.merge(veg_agg, how = 'inner', on = 'plot_id')

In [11]:
# --- NDVI anomaly: plot deviation from vineyard mean per week × year ---
# Strips vintage effects, isolates the spatial (terroir) signal
ndvi_cols = [f'ndvi_smooth_mean_{w}' for w in range(36, 44)]
for col in ndvi_cols:
    yearly_mean = df.groupby('year')[col].transform('mean')
    df[f'ndvi_anomaly_{col.split("_")[-1]}'] = df[col] - yearly_mean

anomaly_cols = [f'ndvi_anomaly_{w}' for w in range(36, 44)]
print('Anomaly range:', df[anomaly_cols].min().min(), 'to', df[anomaly_cols].max().max())
print('Vineyard mean anomaly (should be ~0):', df[anomaly_cols].mean().mean())

Anomaly range: -0.411594403586821 to 0.40475673205779605
Vineyard mean anomaly (should be ~0): -1.4272560343147924e-17


In [12]:
df

,plot_id,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,geometry,total_relief,area_m2,area_ha,aspect_min_cos,aspect_min_sin,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_rad,slope_grad,slope_x,slope_y,year,ndvi_smooth_mean_13,ndvi_smooth_mean_14,ndvi_smooth_mean_15,ndvi_smooth_mean_16,ndvi_smooth_mean_17,ndvi_smooth_mean_18,ndvi_smooth_mean_19,ndvi_smooth_mean_20,ndvi_smooth_mean_21,ndvi_smooth_mean_22,ndvi_smooth_mean_23,ndvi_smooth_mean_24,ndvi_smooth_mean_25,ndvi_smooth_mean_26,ndvi_smooth_mean_27,ndvi_smooth_mean_28,ndvi_smooth_mean_29,ndvi_smooth_mean_30,ndvi_smooth_mean_31,ndvi_smooth_mean_32,ndvi_smooth_mean_33,ndvi_smooth_mean_34,ndvi_smooth_mean_35,ndvi_smooth_mean_36,ndvi_smooth_mean_37,ndvi_smooth_mean_38,ndvi_smooth_mean_39,ndvi_smooth_mean_40,ndvi_smooth_mean_41,ndvi_smooth_mean_42,ndvi_smooth_mean_43,ndvi_smooth_mean_44,ndvi_smooth_mean_45,ndvi_smooth_slope_1,ndvi_smooth_slope_2,ndvi_smooth_slope_3,ndvi_smooth_slope_4,ndvi_smooth_slope_5,ndvi_smooth_slope_6,ndvi_smooth_slope_7,ndvi_smooth_slope_8,ndvi_smooth_slope_9,ndvi_smooth_slope_10,ndvi_smooth_slope_11,ndvi_smooth_slope_12,ndvi_smooth_slope_13,ndvi_smooth_slope_14,ndvi_smooth_slope_15,ndvi_smooth_slope_16,ndvi_smooth_slope_17,ndvi_smooth_slope_18,ndvi_smooth_slope_19,ndvi_smooth_slope_20,ndvi_smooth_slope_21,ndvi_smooth_slope_22,ndvi_smooth_slope_23,ndvi_smooth_slope_24,ndvi_smooth_slope_25,ndvi_smooth_slope_26,ndvi_smooth_slope_27,ndvi_smooth_slope_28,ndvi_smooth_slope_29,ndvi_smooth_slope_30,ndvi_smooth_slope_31,ndvi_smooth_slope_32,ndvi_smooth_slope_33,ndvi_smooth_slope_34,ndvi_smooth_slope_35,ndvi_smooth_slope_36,ndvi_smooth_slope_37,ndvi_smooth_slope_38,ndvi_smooth_slope_39,ndvi_smooth_slope_40,ndvi_smooth_slope_41,ndvi_smooth_slope_42,ndvi_smooth_slope_43,ndvi_smooth_slope_44,ndvi_smooth_slope_45,ndvi_smooth_slope_46,ndvi_smooth_slope_47,ndvi_smooth_slope_48,ndvi_smooth_slope_49,ndvi_smooth_slope_50,ndvi_smooth_slope_51,ndvi_smooth_slope_52,ndvi_smooth_std_1,ndvi_smooth_std_2,ndvi_smooth_std_3,ndvi_smooth_std_4,ndvi_smooth_std_5,ndvi_smooth_std_6,ndvi_smooth_std_7,ndvi_smooth_std_8,ndvi_smooth_std_9,ndvi_smooth_std_10,ndvi_smooth_std_11,ndvi_smooth_std_12,ndvi_smooth_std_13,ndvi_smooth_std_14,ndvi_smooth_std_15,ndvi_smooth_std_16,ndvi_smooth_std_17,ndvi_smooth_std_18,ndvi_smooth_std_19,ndvi_smooth_std_20,ndvi_smooth_std_21,ndvi_smooth_std_22,ndvi_smooth_std_23,ndvi_smooth_std_24,ndvi_smooth_std_25,ndvi_smooth_std_26,ndvi_smooth_std_27,ndvi_smooth_std_28,ndvi_smooth_std_29,ndvi_smooth_std_30,ndvi_smooth_std_31,ndvi_smooth_std_32,ndvi_smooth_std_33,ndvi_smooth_std_34,ndvi_smooth_std_35,ndvi_smooth_std_36,ndvi_smooth_std_37,ndvi_smooth_std_38,ndvi_smooth_std_39,ndvi_smooth_std_40,ndvi_smooth_std_41,ndvi_smooth_std_42,ndvi_smooth_std_43,ndvi_smooth_std_44,ndvi_smooth_std_45,ndvi_smooth_std_46,ndvi_smooth_std_47,ndvi_smooth_std_48,ndvi_smooth_std_49,ndvi_smooth_std_50,ndvi_smooth_std_51,ndvi_smooth_std_52,ndvi_cov,ndvi_mean,ndvi_std,ndvi_mean_norm,ndvi_std_norm,ndvi_cov_norm,health,ndvi_anomaly_36,ndvi_anomaly_37,ndvi_anomaly_38,ndvi_anomaly_39,ndvi_anomaly_40,ndvi_anomaly_41,ndvi_anomaly_42,ndvi_anomaly_43
0,0,0.000489,-0.009361,0.013225,0.000620,-0.010633,0.013191,0.000489,-0.009361,0.013225,207.326294,208.851974,208.022307,22.362457,23.888138,23.058465,"POLYGON ((-119.78208 45.87245, -119.78215 45.8...",1.525681,451.141838,0.045114,-0.909044,0.416701,-0.983593,-0.180402,-0.996750,0.080559,0.087617,0.087841,-0.087556,0.007076,2016,0.417441,0.417741,0.415264,0.412015,0.410159,0.410159,0.408649,0.406006,0.403366,0.400812,0.398296,0.395780,0.393264,0.390748,0.388232,0.384497,0.373238,0.356197,0.348879,0.346145,0.345504,0.355702,0.381068,0.413351,0.438675,0.443236,0.440521,0.432348,0.424176,0.416003,0.407830,0.379300,0.366964,0.0,0.000000,0.000000,0.000000,0.000000,0.000076,0.000076,0.000076,0.00007

### Now let's add some weather information

First load the wather that has been unzipped and clipped to the vineyard

In [13]:
weather = pd.read_pickle('../data/PRISM/df.pkl')

weather = (
    weather
    .groupby("date", as_index=False)
    .first()
)

weather['date'] = pd.to_datetime(weather['date'])
weather['doy'] = weather['date'].dt.dayofyear
weather['year'] = weather['date'].dt.year

weather


,date,ppt,tmax,tmean,tmin,vpdmax,vpdmin,doy,year
0,2016-01-01,0.000000,-4.994952,-6.205034,-7.415116,1.100000,0.440088,1,2016
1,2016-01-02,0.000000,-5.805048,-7.305048,-8.805048,0.700000,0.727885,2,2016
2,2016-01-03,1.000000,-5.505030,-6.977509,-8.449988,0.000000,0.644641,3,2016
3,2016-01-04,0.000000,-3.160078,-4.837600,-6.515122,0.000000,0.000000,4,2016
4,2016-01-05,0.455039,-1.105036,-2.687591,-4.270146,0.000000,0.000000,5,2016
...,...,...,...,...,...,...,...,...,...
3648,2025-12-27,0.000000,6.105038,2.199998,-1.705042,1.800000,0.066595,361,2025
3649,2025-12-28,0.000000,4.950006,0.622482,-3.705042,1.755039,0.000000,362,2025
3650,2025-12-29,0.000000,5.749994,0.949991,-3.850012,2.200000,0.112112,363,2025
3651,2025-12-30,0.000000,4.249994,-0.322484,-4.894962,1.600000,0.000000,364,2025


### Crack at some frost and growing degree days cumulative per month

In [14]:
# Compute frost days and GDD
weather['frost'] = (weather['tmin'] < 0)
weather['gdd'] = (weather['tmean'] - 10).clip(lower=0)


In [15]:
# Compute cumulative GDD for each month
weather['week'] = weather['date'].dt.week
cumulative_gdd = weather.groupby([ 'year', 'week'])['gdd'].sum().groupby(level=[0,1]).cumsum()
# Reset index to turn MultiIndex into columns
cumulative_gdd = cumulative_gdd.rename('cumulative_gdd').reset_index()

# Merge back to weather
weather = weather.merge(
    cumulative_gdd,
    on=['year', 'week'],
    how='left'
)


weather

/home/simonhans/anaconda3/envs/GrapeExpectations/lib/python3.7/site-packages/ipykernel_launcher.py:2: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated.  Please use Series.dt.isocalendar().week instead.
  


,date,ppt,tmax,tmean,tmin,vpdmax,vpdmin,doy,year,frost,gdd,week,cumulative_gdd
0,2016-01-01,0.000000,-4.994952,-6.205034,-7.415116,1.100000,0.440088,1,2016,True,0.0,53,0.0
1,2016-01-02,0.000000,-5.805048,-7.305048,-8.805048,0.700000,0.727885,2,2016,True,0.0,53,0.0
2,2016-01-03,1.000000,-5.505030,-6.977509,-8.449988,0.000000,0.644641,3,2016,True,0.0,53,0.0
3,2016-01-04,0.000000,-3.160078,-4.837600,-6.515122,0.000000,0.000000,4,2016,True,0.0,1,0.0
4,2016-01-05,0.455039,-1.105036,-2.687591,-4.270146,0.000000,0.000000,5,2016,True,0.0,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3648,2025-12-27,0.000000,6.105038,2.199998,-1.705042,1.800000,0.066595,361,2025,True,0.0,52,0.0
3649,2025-12-28,0.000000,4.950006,0.622482,-3.705042,1.755039,0.000000,362,2025,True,0.0,52,0.0
3650,2025-12-29,0.000000,5.749994,0.949991,-3.850012,2.200000,0.112112,363,2025,True,0.0,1,0.0
3651,2025-12-30,0.000000,4.249994,-0.322484,-4.894962,1.600000,0.000000,364,2025,True,0.0,1,0.0


In [16]:
weather = weather[
    (weather['week'] > 27) &
    (weather['week'] < 44)
].copy()

In [17]:
# Define which aggregations to use
agg_funcs = {
    'ppt': 'sum',
    'tmax': 'max',
    'tmin': 'min',
    'tmean': 'mean',
    'vpdmax': 'max',
    'vpdmin': 'min',
    'cumulative_gdd': 'max'
}

# Aggregate to long-form first (year, week, variables)
weekly_long = weather.groupby(['year','week']).agg(agg_funcs).reset_index()

# Pivot to wide form (one row per year, columns per week)
weekly_wide = pd.DataFrame({'year': weekly_long['year'].unique()})

for col in ['ppt','tmax','tmin','tmean','vpdmax','vpdmin','cumulative_gdd']:
    pivoted = weekly_long.pivot(index='year', columns='week', values=col)
    # Rename columns to include variable name
    pivoted.columns = [f'{col}_{w}' for w in pivoted.columns]
    weekly_wide = weekly_wide.merge(pivoted, on='year', how='left')

weekly_wide.reset_index(drop=True, inplace=True)

In [18]:
weekly_wide

,year,ppt_28,ppt_29,ppt_30,ppt_31,ppt_32,ppt_33,ppt_34,ppt_35,ppt_36,ppt_37,ppt_38,ppt_39,ppt_40,ppt_41,ppt_42,ppt_43,tmax_28,tmax_29,tmax_30,tmax_31,tmax_32,tmax_33,tmax_34,tmax_35,tmax_36,tmax_37,tmax_38,tmax_39,tmax_40,tmax_41,tmax_42,tmax_43,tmin_28,tmin_29,tmin_30,tmin_31,tmin_32,tmin_33,tmin_34,tmin_35,tmin_36,tmin_37,tmin_38,tmin_39,tmin_40,tmin_41,tmin_42,tmin_43,tmean_28,tmean_29,tmean_30,tmean_31,tmean_32,tmean_33,tmean_34,tmean_35,tmean_36,tmean_37,tmean_38,tmean_39,tmean_40,tmean_41,tmean_42,tmean_43,vpdmax_28,vpdmax_29,vpdmax_30,vpdmax_31,vpdmax_32,vpdmax_33,vpdmax_34,vpdmax_35,vpdmax_36,vpdmax_37,vpdmax_38,vpdmax_39,vpdmax_40,vpdmax_41,vpdmax_42,vpdmax_43,vpdmin_28,vpdmin_29,vpdmin_30,vpdmin_31,vpdmin_32,vpdmin_33,vpdmin_34,vpdmin_35,vpdmin_36,vpdmin_37,vpdmin_38,vpdmin_39,vpdmin_40,vpdmin_41,vpdmin_42,vpdmin_43,cumulative_gdd_28,cumulative_gdd_29,cumulative_gdd_30,cumulative_gdd_31,cumulative_gdd_32,cumulative_gdd_33,cumulative_gdd_34,cumulative_gdd_35,cumulative_gdd_36,cumulative_gdd_37,cumulative_gdd_38,cumulative_gdd_39,cumulative_gdd_40,cumulative_gdd_41,cumulative_gdd_42,cumulative_gdd_43
0,2016,0.000000,1.310078,0.000000,0.0,1.365116,0.000000,0.000000,0.00000,5.789922,1.385271,0.000000,0.000000,6.185271,18.615504,4.020155,21.570543,33.639926,36.139926,38.774818,35.305036,36.529842,36.739916,32.894964,33.339922,32.194952,31.160080,27.194952,29.350000,24.050012,20.305036,17.494970,17.774818,13.105038,13.294958,12.884886,11.474800,12.384886,12.429836,10.639926,7.760072,7.615116,4.370156,5.594962,6.115116,6.460084,0.205030,2.794958,4.550012,21.944242,22.429568,26.234941,21.894516,22.867052,24.830620,21.519602,19.065004,18.032170,17.391863,14.584660,17.102157,14.042137,10.906034,10.615336,10.066391,23.099999,24.934884,34.349613,26.010077,27.969767,29.644962,23.610076,23.644960,19.455039,15.065116,12.200000,15.810078,7.844961,7.444961,5.655039,4.700000,1.881505,1.950429,5.053060,1.712215,0.265490,4.835625,2.866083,0.850186,0.000000,0.000000,0.000000,1.087746,0.000000,0.000000,0.0,0.0,83.609693,87.006976,113.644589,83.261614,90.069361,103.814337,80.637215,63.455025,56.225192,51.743038,32.092622,49.715100,28.294959,10.842245,5.222467,5.369773
1,2017,0.000000,0.000000,0.000000,0.0,1.955039,0.000000,0.000000,0.00000,0.000000,0.644961,3.500000,0.165116,0.000000,1.220155,23.010078,0.000000,34.539920,37.739916,36.450006,38.194952,38.850000,33.994970,34.149988,36.694952,32.460084,33.705030,22.905042,28.394964,23.694952,17.784880,21.394964,20.294958,11.729854,11.294958,14.050012,14.950006,12.950006,10.539920,9.739916,13.560074,11.215122,5.860078,3.515110,8.394964,2.305036,-0.815110,0.015110,1.994970,23.469933,23.499556,25.381009,26.465334,26.314228,21.716722,21.802823,24.371096,22.391474,17.965728,13.128242,16.541088,12.202159,8.616778,11.253931,10.224969,25.044961,28.244961,29.900000,32.355039,33.810078,23.044961,26.310078,30.200000,23.975194,23.320154,9.610078,14.355039,12.800000,7.955039,9.120155,6.044961,1.874487,3.739983,5.032998,6.032692,0.000000,2.920158,0.000000,4.213529,0.000000,2.647971,0.000000,0.157001,0.367370,0.000000,0.0,0.0,94.289530,94.496891,107.667065,115.257339,114.199593,82.017053,82.619761,100.597675,86.740318,55.760098,21.897693,45.787616,15.415114,0.932563,9.050005,3.857346
2,2018,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.555039,0.00000,0.000000,0.000000,0.000000,0.000000,8.930232,9.785271,0.000000,1.410078,38.839922,38.284880,38.360078,39.005048,39.294958,36.484878,32.894964,32.894964,33.415120,25.249994,25.094962,27.794958,23.484878,21.484878,18.994970,19.615116,13.094962,11.950006,12.715122,12.229854,12.474800,12.139926,10.384886,9.360078,10.094962,6.460084,5.505048,4.450006,3.249994,0.705030,0.160080,2.105038,24.413899,24.148504,26.583888,24.722038,25.505977,23.633835,20.749946,19.454266,20.178242,16.018936,14.948948,15.640699,11.288924,10.804261,9.663566,12.058583,32.844961,34.579845,32.920155,34.775194,35.189922,30.114729,22.744961,21.355040,22.575194,12.275194,13.410078,17.655039,1

### Now we can combine data.

In [19]:
df['year'] = df['year'].astype(int)
df = pd.merge(df, weekly_wide, how = 'inner', on = 'year')

In [20]:
df

,plot_id,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,geometry,total_relief,area_m2,area_ha,aspect_min_cos,aspect_min_sin,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_rad,slope_grad,slope_x,slope_y,year,ndvi_smooth_mean_13,ndvi_smooth_mean_14,ndvi_smooth_mean_15,ndvi_smooth_mean_16,ndvi_smooth_mean_17,ndvi_smooth_mean_18,ndvi_smooth_mean_19,ndvi_smooth_mean_20,ndvi_smooth_mean_21,ndvi_smooth_mean_22,ndvi_smooth_mean_23,ndvi_smooth_mean_24,ndvi_smooth_mean_25,ndvi_smooth_mean_26,ndvi_smooth_mean_27,ndvi_smooth_mean_28,ndvi_smooth_mean_29,ndvi_smooth_mean_30,ndvi_smooth_mean_31,ndvi_smooth_mean_32,ndvi_smooth_mean_33,ndvi_smooth_mean_34,ndvi_smooth_mean_35,ndvi_smooth_mean_36,ndvi_smooth_mean_37,ndvi_smooth_mean_38,ndvi_smooth_mean_39,ndvi_smooth_mean_40,ndvi_smooth_mean_41,ndvi_smooth_mean_42,ndvi_smooth_mean_43,ndvi_smooth_mean_44,ndvi_smooth_mean_45,ndvi_smooth_slope_1,ndvi_smooth_slope_2,ndvi_smooth_slope_3,ndvi_smooth_slope_4,ndvi_smooth_slope_5,ndvi_smooth_slope_6,ndvi_smooth_slope_7,ndvi_smooth_slope_8,ndvi_smooth_slope_9,ndvi_smooth_slope_10,ndvi_smooth_slope_11,ndvi_smooth_slope_12,ndvi_smooth_slope_13,ndvi_smooth_slope_14,ndvi_smooth_slope_15,ndvi_smooth_slope_16,ndvi_smooth_slope_17,ndvi_smooth_slope_18,ndvi_smooth_slope_19,ndvi_smooth_slope_20,ndvi_smooth_slope_21,ndvi_smooth_slope_22,ndvi_smooth_slope_23,ndvi_smooth_slope_24,ndvi_smooth_slope_25,ndvi_smooth_slope_26,ndvi_smooth_slope_27,ndvi_smooth_slope_28,ndvi_smooth_slope_29,ndvi_smooth_slope_30,ndvi_smooth_slope_31,ndvi_smooth_slope_32,ndvi_smooth_slope_33,ndvi_smooth_slope_34,ndvi_smooth_slope_35,ndvi_smooth_slope_36,ndvi_smooth_slope_37,ndvi_smooth_slope_38,ndvi_smooth_slope_39,ndvi_smooth_slope_40,ndvi_smooth_slope_41,ndvi_smooth_slope_42,ndvi_smooth_slope_43,ndvi_smooth_slope_44,ndvi_smooth_slope_45,ndvi_smooth_slope_46,ndvi_smooth_slope_47,ndvi_smooth_slope_48,ndvi_smooth_slope_49,ndvi_smooth_slope_50,ndvi_smooth_slope_51,ndvi_smooth_slope_52,ndvi_smooth_std_1,ndvi_smooth_std_2,ndvi_smooth_std_3,ndvi_smooth_std_4,ndvi_smooth_std_5,ndvi_smooth_std_6,ndvi_smooth_std_7,ndvi_smooth_std_8,ndvi_smooth_std_9,ndvi_smooth_std_10,ndvi_smooth_std_11,ndvi_smooth_std_12,ndvi_smooth_std_13,ndvi_smooth_std_14,ndvi_smooth_std_15,ndvi_smooth_std_16,ndvi_smooth_std_17,ndvi_smooth_std_18,ndvi_smooth_std_19,ndvi_smooth_std_20,ndvi_smooth_std_21,ndvi_smooth_std_22,ndvi_smooth_std_23,ndvi_smooth_std_24,ndvi_smooth_std_25,ndvi_smooth_std_26,ndvi_smooth_std_27,ndvi_smooth_std_28,ndvi_smooth_std_29,ndvi_smooth_std_30,ndvi_smooth_std_31,ndvi_smooth_std_32,ndvi_smooth_std_33,ndvi_smooth_std_34,ndvi_smooth_std_35,ndvi_smooth_std_36,ndvi_smooth_std_37,ndvi_smooth_std_38,ndvi_smooth_std_39,ndvi_smooth_std_40,ndvi_smooth_std_41,ndvi_smooth_std_42,ndvi_smooth_std_43,ndvi_smooth_std_44,ndvi_smooth_std_45,ndvi_smooth_std_46,ndvi_smooth_std_47,ndvi_smooth_std_48,ndvi_smooth_std_49,ndvi_smooth_std_50,ndvi_smooth_std_51,ndvi_smooth_std_52,ndvi_cov,ndvi_mean,ndvi_std,ndvi_mean_norm,ndvi_std_norm,ndvi_cov_norm,health,ndvi_anomaly_36,ndvi_anomaly_37,ndvi_anomaly_38,ndvi_anomaly_39,ndvi_anomaly_40,ndvi_anomaly_41,ndvi_anomaly_42,ndvi_anomaly_43,ppt_28,ppt_29,ppt_30,ppt_31,ppt_32,ppt_33,ppt_34,ppt_35,ppt_36,ppt_37,ppt_38,ppt_39,ppt_40,ppt_41,ppt_42,ppt_43,tmax_28,tmax_29,tmax_30,tmax_31,tmax_32,tmax_33,tmax_34,tmax_35,tmax_36,tmax_37,tmax_38,tmax_39,tmax_40,tmax_41,tmax_42,tmax_43,tmin_28,tmin_29,tmin_30,tmin_31,tmin_32,tmin_33,tmin_34,tmin_35,tmin_36,tmin_37,tmin_38,tmin_39,tmin_40,tmin_41,tmin_42,tmin_43,tmean_28,tmean_29,tmean_30,tmean_31,tmean_32,tmean_33,tmean_34,tmean_35,tmean_36,tmean_37,tmean_38,tmean_39,tmean_40,tmean_41,tmean_42,tmean_43,vpdmax_28,vpdmax_29,vpdmax_30,vpdmax_31,vpdmax_32,vpdmax_33,vpdmax_34,vpdmax_35,vpdmax_36,vpdmax_37,vpdmax_38,vpdmax_39,vpdmax_40,vpdmax_41,vpdmax_42,vpdmax_43,vpdmin_28,vpdmin_29,vpdmin_30,v

In [21]:
# # Optional: could also add slope magnitude transforms
# df['slope_squared'] = df['slope_mean'] ** 2
# df['slope_log'] = np.log1p(df['slope_mean'])  # log(1 + slope)

for i in range(28, 44, 1):
    
    df[f'water_availability_{i}'] = df[f'ppt_{i}'] / (1 +  df[f'cumulative_gdd_{i}'])
    df[f'diurnal_temp_range_{i}'] = df[f'tmax_{i}'] / df[f'tmin_{i}']
    df[f'stress_index{i}'] = df[f'vpdmax_{i}'] / (df[f'ppt_{i}'] + 0.1)
df['local_relief'] = df['elev_mean'] - df['elev_min']

df['total_relief_log'] = np.log1p(df['total_relief'])

In [22]:
soil = pd.read_pickle('../data/soil/plot_summary.pkl')

In [23]:
df = pd.merge(df, soil, how = 'inner', on = 'plot_id')

In [24]:
df = pd.DataFrame(df.drop(columns = 'geometry'))

In [25]:
df

,plot_id,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,total_relief,area_m2,area_ha,aspect_min_cos,aspect_min_sin,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_rad,slope_grad,slope_x,slope_y,year,ndvi_smooth_mean_13,ndvi_smooth_mean_14,ndvi_smooth_mean_15,ndvi_smooth_mean_16,ndvi_smooth_mean_17,ndvi_smooth_mean_18,ndvi_smooth_mean_19,ndvi_smooth_mean_20,ndvi_smooth_mean_21,ndvi_smooth_mean_22,ndvi_smooth_mean_23,ndvi_smooth_mean_24,ndvi_smooth_mean_25,ndvi_smooth_mean_26,ndvi_smooth_mean_27,ndvi_smooth_mean_28,ndvi_smooth_mean_29,ndvi_smooth_mean_30,ndvi_smooth_mean_31,ndvi_smooth_mean_32,ndvi_smooth_mean_33,ndvi_smooth_mean_34,ndvi_smooth_mean_35,ndvi_smooth_mean_36,ndvi_smooth_mean_37,ndvi_smooth_mean_38,ndvi_smooth_mean_39,ndvi_smooth_mean_40,ndvi_smooth_mean_41,ndvi_smooth_mean_42,ndvi_smooth_mean_43,ndvi_smooth_mean_44,ndvi_smooth_mean_45,ndvi_smooth_slope_1,ndvi_smooth_slope_2,ndvi_smooth_slope_3,ndvi_smooth_slope_4,ndvi_smooth_slope_5,ndvi_smooth_slope_6,ndvi_smooth_slope_7,ndvi_smooth_slope_8,ndvi_smooth_slope_9,ndvi_smooth_slope_10,ndvi_smooth_slope_11,ndvi_smooth_slope_12,ndvi_smooth_slope_13,ndvi_smooth_slope_14,ndvi_smooth_slope_15,ndvi_smooth_slope_16,ndvi_smooth_slope_17,ndvi_smooth_slope_18,ndvi_smooth_slope_19,ndvi_smooth_slope_20,ndvi_smooth_slope_21,ndvi_smooth_slope_22,ndvi_smooth_slope_23,ndvi_smooth_slope_24,ndvi_smooth_slope_25,ndvi_smooth_slope_26,ndvi_smooth_slope_27,ndvi_smooth_slope_28,ndvi_smooth_slope_29,ndvi_smooth_slope_30,ndvi_smooth_slope_31,ndvi_smooth_slope_32,ndvi_smooth_slope_33,ndvi_smooth_slope_34,ndvi_smooth_slope_35,ndvi_smooth_slope_36,ndvi_smooth_slope_37,ndvi_smooth_slope_38,ndvi_smooth_slope_39,ndvi_smooth_slope_40,ndvi_smooth_slope_41,ndvi_smooth_slope_42,ndvi_smooth_slope_43,ndvi_smooth_slope_44,ndvi_smooth_slope_45,ndvi_smooth_slope_46,ndvi_smooth_slope_47,ndvi_smooth_slope_48,ndvi_smooth_slope_49,ndvi_smooth_slope_50,ndvi_smooth_slope_51,ndvi_smooth_slope_52,ndvi_smooth_std_1,ndvi_smooth_std_2,ndvi_smooth_std_3,ndvi_smooth_std_4,ndvi_smooth_std_5,ndvi_smooth_std_6,ndvi_smooth_std_7,ndvi_smooth_std_8,ndvi_smooth_std_9,ndvi_smooth_std_10,ndvi_smooth_std_11,ndvi_smooth_std_12,ndvi_smooth_std_13,ndvi_smooth_std_14,ndvi_smooth_std_15,ndvi_smooth_std_16,ndvi_smooth_std_17,ndvi_smooth_std_18,ndvi_smooth_std_19,ndvi_smooth_std_20,ndvi_smooth_std_21,ndvi_smooth_std_22,ndvi_smooth_std_23,ndvi_smooth_std_24,ndvi_smooth_std_25,ndvi_smooth_std_26,ndvi_smooth_std_27,ndvi_smooth_std_28,ndvi_smooth_std_29,ndvi_smooth_std_30,ndvi_smooth_std_31,ndvi_smooth_std_32,ndvi_smooth_std_33,ndvi_smooth_std_34,ndvi_smooth_std_35,ndvi_smooth_std_36,ndvi_smooth_std_37,ndvi_smooth_std_38,ndvi_smooth_std_39,ndvi_smooth_std_40,ndvi_smooth_std_41,ndvi_smooth_std_42,ndvi_smooth_std_43,ndvi_smooth_std_44,ndvi_smooth_std_45,ndvi_smooth_std_46,ndvi_smooth_std_47,ndvi_smooth_std_48,ndvi_smooth_std_49,ndvi_smooth_std_50,ndvi_smooth_std_51,ndvi_smooth_std_52,ndvi_cov,ndvi_mean,ndvi_std,ndvi_mean_norm,ndvi_std_norm,ndvi_cov_norm,health,ndvi_anomaly_36,ndvi_anomaly_37,ndvi_anomaly_38,ndvi_anomaly_39,ndvi_anomaly_40,ndvi_anomaly_41,ndvi_anomaly_42,ndvi_anomaly_43,ppt_28,ppt_29,ppt_30,ppt_31,ppt_32,ppt_33,ppt_34,ppt_35,ppt_36,ppt_37,ppt_38,ppt_39,ppt_40,ppt_41,ppt_42,ppt_43,tmax_28,tmax_29,tmax_30,tmax_31,tmax_32,tmax_33,tmax_34,tmax_35,tmax_36,tmax_37,tmax_38,tmax_39,tmax_40,tmax_41,tmax_42,tmax_43,tmin_28,tmin_29,tmin_30,tmin_31,tmin_32,tmin_33,tmin_34,tmin_35,tmin_36,tmin_37,tmin_38,tmin_39,tmin_40,tmin_41,tmin_42,tmin_43,tmean_28,tmean_29,tmean_30,tmean_31,tmean_32,tmean_33,tmean_34,tmean_35,tmean_36,tmean_37,tmean_38,tmean_39,tmean_40,tmean_41,tmean_42,tmean_43,vpdmax_28,vpdmax_29,vpdmax_30,vpdmax_31,vpdmax_32,vpdmax_33,vpdmax_34,vpdmax_35,vpdmax_36,vpdmax_37,vpdmax_38,vpdmax_39,vpdmax_40,vpdmax_41,vpdmax_42,vpdmax_43,vpdmin_28,vpdmin_29,vpdmin_30,vpdmin_31,

In [26]:
df.to_pickle('../data/df.pkl')